In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_sparse import SparseTensor
from typing import Tuple, Optional, Union
import math

In [14]:
import torch
import torch.nn as nn
from torch_sparse import SparseTensor
from typing import Tuple, Optional, Union
import math

class SparseConv3d(nn.Module):
    """
    Sparse 3D Convolution layer using torch_sparse backend.
    Efficiently implements convolution using SparseTensor for sparse operations.
    """
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: Union[int, Tuple[int, int, int]],
        stride: Union[int, Tuple[int, int, int]] = 1,
        padding: Union[int, Tuple[int, int, int]] = 0,
        dilation: Union[int, Tuple[int, int, int]] = 1,
        bias: bool = True
    ):
        super().__init__()
        
        # Handle different input formats
        if isinstance(kernel_size, int):
            kernel_size = (kernel_size,) * 3
        if isinstance(stride, int):
            stride = (stride,) * 3
        if isinstance(padding, int):
            padding = (padding,) * 3
        if isinstance(dilation, int):
            dilation = (dilation,) * 3
            
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding
        self.dilation = dilation
        
        # Initialize weights using Kaiming initialization
        self.weight = nn.Parameter(
            torch.empty(out_channels, in_channels, *kernel_size)
        )
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        
        if bias:
            self.bias = nn.Parameter(torch.empty(out_channels))
            fan_in = in_channels * (kernel_size[0] * kernel_size[1] * kernel_size[2])
            bound = 1 / math.sqrt(fan_in)
            nn.init.uniform_(self.bias, -bound, bound)
        else:
            self.register_parameter('bias', None)

    def _get_kernel_offsets(self) -> torch.Tensor:
        """Generate kernel offsets for the convolution."""
        offsets = []
        for z in range(self.kernel_size[0]):
            for y in range(self.kernel_size[1]):
                for x in range(self.kernel_size[2]):
                    offsets.append([z, y, x])
        return torch.tensor(offsets, dtype=torch.long)

    def _create_sparse_tensor(
        self,
        coords: torch.Tensor,
        features: torch.Tensor,
        spatial_shape: Tuple[int, int, int]
    ) -> SparseTensor:
        """Create a SparseTensor from coordinates and features."""
        row = coords[:, 0] * (spatial_shape[1] * spatial_shape[2]) + \
              coords[:, 1] * spatial_shape[2] + \
              coords[:, 2]
        
        return SparseTensor(
            row=row,
            col=torch.arange(row.size(0), device=coords.device),
            value=features,
            sparse_sizes=(spatial_shape[0] * spatial_shape[1] * spatial_shape[2],
                         features.size(0))
        )

    def forward(
        self,
        x: torch.Tensor,
        coords: torch.Tensor,
        batch: Optional[torch.Tensor] = None,
        spatial_shape: Optional[Tuple[int, int, int]] = None
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Forward pass of sparse convolution using SparseTensor.
        
        Args:
            x: (N, C_in) Feature matrix
            coords: (N, 3) Coordinate matrix
            batch: (N,) Batch vector, optional
            spatial_shape: Tuple of spatial dimensions (optional)
        """
        device = x.device
        
        # Handle padding
        if any(p != 0 for p in self.padding):
            coords = coords + torch.tensor(self.padding, 
                                        device=device, 
                                        dtype=coords.dtype).view(1, 3)
        
        if batch is None:
            batch = torch.zeros(coords.size(0), dtype=torch.long, device=device)
        
        if spatial_shape is None:
            spatial_shape = tuple(
                coords[:, i].max().item() + 1 for i in range(3)
            )

        # Get kernel offsets
        kernel_offsets = self._get_kernel_offsets().to(device)
        kernel_offsets = kernel_offsets * torch.tensor(self.dilation, 
                                                     device=device).view(1, 3)

        # Create output feature container
        output_features_dict = {}
        output_coords_set = set()

        # For each kernel position
        for k_idx, offset in enumerate(kernel_offsets):
            # Calculate neighbor coordinates
            neighbor_coords = coords + offset.view(1, 3)
            
            # Check bounds
            valid_mask = (
                (neighbor_coords >= 0).all(dim=1) & 
                (neighbor_coords < torch.tensor(spatial_shape, device=device)).all(dim=1)
            )
            
            if not valid_mask.any():
                continue

            # Get valid neighbors
            valid_coords = neighbor_coords[valid_mask]
            valid_features = x[valid_mask]

            # Convert to linear indices
            linear_indices = (valid_coords[:, 0] * (spatial_shape[1] * spatial_shape[2]) + 
                            valid_coords[:, 1] * spatial_shape[2] + 
                            valid_coords[:, 2])

            # Get weight for this kernel position
            k_weight = self.weight[:, :, 
                                 k_idx // (self.kernel_size[1] * self.kernel_size[2]),
                                 (k_idx % (self.kernel_size[1] * self.kernel_size[2])) // self.kernel_size[2],
                                 k_idx % self.kernel_size[2]].t()  # (C_in, C_out)

            # Compute features for this kernel position
            out_features = torch.mm(valid_features, k_weight)  # (N, C_out)

            # Accumulate features
            for idx, linear_idx in enumerate(linear_indices):
                linear_idx = linear_idx.item()
                if linear_idx in output_features_dict:
                    output_features_dict[linear_idx] += out_features[idx]
                else:
                    output_features_dict[linear_idx] = out_features[idx]
                output_coords_set.add(linear_idx)

        if not output_features_dict:
            return torch.empty(0, self.out_channels, device=device), \
                   torch.empty(0, 3, device=device)

        # Convert accumulated features to tensor
        output_indices = sorted(output_coords_set)
        output_features = torch.stack([output_features_dict[idx] for idx in output_indices])

        # Convert linear indices back to 3D coordinates
        output_coords = torch.stack([
            torch.div(torch.tensor(output_indices, device=device),
                     spatial_shape[1] * spatial_shape[2], rounding_mode='floor'),
            torch.div(torch.tensor(output_indices, device=device) % (spatial_shape[1] * spatial_shape[2]),
                     spatial_shape[2], rounding_mode='floor'),
            torch.tensor(output_indices, device=device) % spatial_shape[2]
        ], dim=1)

        # Add bias if present
        if self.bias is not None:
            output_features += self.bias

        return output_features, output_coords

    def extra_repr(self) -> str:
        """String representation of the module."""
        s = (f'{self.in_channels}, {self.out_channels}, kernel_size={self.kernel_size}'
             f', stride={self.stride}')
        if self.padding != (0,) * 3:
            s += f', padding={self.padding}'
        if self.dilation != (1,) * 3:
            s += f', dilation={self.dilation}'
        if self.bias is None:
            s += ', bias=False'
        return s

In [15]:
conv = SparseConv3d(
    in_channels=64,
    out_channels=128,
    kernel_size=3,
    stride=1,
    padding=1
)

# Forward pass
features = torch.randn(1000, 64)  # 1000 points with 64 features each
coords = torch.randint(0, 32, (1000, 3))  # 3D coordinates
output_features, output_coords = conv(features, coords)

In [ ]:
device = "cpu"


conv = SparseConv3d(
    in_channels=1,
    out_channels=1,
    kernel_size=3,
    padding=1
).to(device)

# Set only the center weight to 1 (identity mapping)
with torch.no_grad():
    conv.weight.fill_(0.0)
    # Set the center of the kernel to 1
    conv.weight[0, 0, 1, 1, 1] = 1.0
    conv.bias.fill_(0.0)

# Create a single point with feature value 1
x = torch.ones(1, 1).to(device)
coords = torch.tensor([[5, 5, 5]], device=device)

output_features, output_coords = conv(x, coords)

# The output should be exactly 1 due to identity mapping
torch.allclose(output_features, torch.tensor([[1.0]], device=device),  rtol=1e-5)

False

In [26]:
output_features

tensor([[0.]], grad_fn=<AddBackward0>)